<a href="https://colab.research.google.com/github/MalshanRuchira/NorthStar-Analytics-Project/blob/main/NorthStar_Main_Analysis_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
install.packages("RSQLite", dependencies=TRUE)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
install.packages("sqldf")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [5]:
library(sqldf)

orders <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/orders.csv")
deliveries <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/deliveries.csv")
vehicles <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/vehicles.csv")
incidents <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/incidents.csv")
drivers <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/drivers.csv")
hubs <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/hubs.csv")
complaints <- read.csv("https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/complaints.csv")

print("Library sqldf loaded successfully!")

[1] "Library sqldf loaded successfully!"


In [11]:
sqldf("SELECT o.service_type, COUNT(d.delivery_id) as total_deliveries,
      SUM(o.order_value) as total_revenue, SUM(d.fuel_or_charge_cost) as total_fuel_cost,
      (SUM(o.order_value) - SUM(d.fuel_or_charge_cost)) as gross_margin
      FROM orders o JOIN deliveries d ON o.order_id = d.order_id
      GROUP BY o.service_type ORDER BY gross_margin ASC")

service_type,total_deliveries,total_revenue,total_fuel_cost,gross_margin
<chr>,<int>,<dbl>,<dbl>,<dbl>
Medical,108,9344.88,1379.48,7965.40
Business,126,12279.23,1655.91,10623.32
Retail,224,19444.86,2906.27,16538.59
Parcel,230,20735.44,3009.01,17726.43
Passenger,262,25463.36,3248.56,22214.80


In [12]:
sqldf("SELECT v.maintenance_status, COUNT(i.incident_id) as total_incidents,
      AVG(v.battery_health_pct) as avg_battery_health, AVG(i.resolved_hours) as avg_resolution_time
      FROM vehicles v JOIN deliveries d ON v.vehicle_id = d.vehicle_id
      JOIN incidents i ON d.delivery_id = i.delivery_id
      GROUP BY v.maintenance_status ORDER BY total_incidents DESC")

maintenance_status,total_incidents,avg_battery_health,avg_resolution_time
<chr>,<int>,<dbl>,<dbl>
Active,150,75.42759,11.58521
InRepair,84,74.91905,12.78481
Scheduled,46,80.89783,11.99762


In [14]:
sqldf("SELECT dr.years_experience, COUNT(d.delivery_id) as total_routes,
      AVG(d.manual_route_override_count) as avg_overrides, AVG(d.customer_rating_post_delivery) as avg_rating
      FROM drivers dr JOIN deliveries d ON dr.driver_id = d.driver_id
      GROUP BY dr.years_experience ORDER BY dr.years_experience ASC")

years_experience,total_routes,avg_overrides,avg_rating
<int>,<int>,<dbl>,<dbl>
1,42,0.8809524,4.012857
2,62,1.0645161,4.027419
3,51,1.0196078,3.719600
4,76,1.0131579,3.730000
5,52,0.8269231,3.935490
6,53,0.9056604,3.863462
7,58,0.9310345,4.078947
8,79,0.9367089,3.807792
9,58,1.2241379,3.990702


In [15]:
sqldf("SELECT h.hub_name, h.hub_type, COUNT(d.delivery_id) as total_dispatch,
      SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed_deliveries,
      SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) as delayed_deliveries
      FROM hubs h JOIN deliveries d ON h.hub_id = d.hub_id
      GROUP BY h.hub_name ORDER BY failed_deliveries DESC")

hub_name,hub_type,total_dispatch,failed_deliveries,delayed_deliveries
<chr>,<chr>,<int>,<int>,<int>
Midtown Relay,Charging,128,26,22
Central Core,Control,115,23,25
North Exchange,Dispatch,136,17,26
West Gate,Dispatch,127,16,28
Airport Hub,Dispatch,104,15,27
Riverside Hub,Warehouse,115,14,25
East Dock,Warehouse,119,11,23
South Link,Dispatch,106,10,26


In [16]:
sqldf("SELECT c.complaint_type, COUNT(c.complaint_id) as complaint_count,
      SUM(c.compensation_amount) as total_compensation_paid, AVG(c.resolution_days) as avg_resolution_days
      FROM complaints c JOIN orders o ON c.order_id = o.order_id
      GROUP BY c.complaint_type ORDER BY total_compensation_paid DESC")

complaint_type,complaint_count,total_compensation_paid,avg_resolution_days
<chr>,<int>,<dbl>,<dbl>
Delay,101,1696.84,7.257426
MissedPickup,64,1423.40,7.640625
AppIssue,53,980.72,8.603774
DriverBehaviour,51,973.06,8.156863
Billing,16,381.94,7.750000
Damage,15,359.73,11.333333
SupportExperience,20,342.50,7.450000
